# カメラ外部キャリブレーション 実践（OpenCV）

`extrinsic_calibration.md` の手順を **OpenCV の実API** で実行する。
実画像は不要で、**合成チェッカーボードの対応点**を生成して自己完結で動く。

1. `calibrateCamera` で内部パラメータ（K, 歪み）を推定
2. `solvePnP` で1台の外部パラメータ `[R|t]` を推定
3. `stereoCalibrate` で2カメラ間の相対 `R,T`
4. `stereoRectify` でレクティフィケーション（R1,R2,P1,P2,Q）
5. 視差→深度（Q による再投影）

> 原理は `extrinsic_calibration_demo.ipynb`（numpy）で先に掴むこと。

In [ ]:
# %pip install opencv-python numpy matplotlib
import cv2, numpy as np, matplotlib.pyplot as plt
np.set_printoptions(precision=4, suppress=True); np.random.seed(0)
print('OpenCV', cv2.__version__)

---

## 0. 合成データ生成

チェッカーボード（9×6内角）を複数姿勢で配置し、既知のK・歪みで投影して `imagePoints` を作る。
`cv2.projectPoints` が歪みを含めて投影してくれる。

In [ ]:
PATTERN=(9,6); SQUARE=0.025          # 25mm マス
IMG_W,IMG_H=1280,720; IMG_SIZE=(IMG_W,IMG_H)

# board座標系の3D点（z=0平面）
objp=np.zeros((PATTERN[0]*PATTERN[1],3),np.float32)
objp[:,:2]=np.mgrid[0:PATTERN[0],0:PATTERN[1]].T.reshape(-1,2)*SQUARE

# 真の内部パラメータ（これを推定で当てにいく）
K_gt=np.array([[800,0,640],[0,800,360]],np.float64)
K_gt=np.array([[800,0,640],[0,800,360],[0,0,1]],np.float64)
dist_gt=np.array([0.05,-0.08,0.001,0.0005,0.0],np.float64)  # k1,k2,p1,p2,k3

def random_board_pose():
    rvec=np.array([np.random.uniform(-0.4,0.4),np.random.uniform(-0.4,0.4),
                   np.random.uniform(-0.3,0.3)])
    tvec=np.array([np.random.uniform(-0.15,0.15),np.random.uniform(-0.1,0.1),
                   np.random.uniform(0.6,1.0)])           # 前方0.6-1.0m
    return rvec.reshape(3,1),tvec.reshape(3,1)

N_VIEWS=18
poses=[random_board_pose() for _ in range(N_VIEWS)]
objpoints=[]; imgpoints=[]
for rvec,tvec in poses:
    img,_=cv2.projectPoints(objp,rvec,tvec,K_gt,dist_gt)
    objpoints.append(objp.copy())
    imgpoints.append(img.reshape(-1,1,2).astype(np.float32))
print(f'{N_VIEWS} 姿勢分の対応点を生成。1姿勢あたり {len(objp)} 点')

---

## 1. 内部キャリブレーション `calibrateCamera`

対応点から K と歪みを推定し、真値と比較する（Zhang法）。

In [ ]:
ret,K_est,dist_est,rvecs,tvecs=cv2.calibrateCamera(
    objpoints,imgpoints,IMG_SIZE,None,None)
print('再投影RMS誤差:',round(ret,5),'px')
print('K 真値:\n',K_gt)
print('K 推定:\n',K_est)
print('dist 真値:',dist_gt)
print('dist 推定:',dist_est.ravel())

---

## 2. 外部パラメータ `solvePnP`

K既知のもとで、1ビューの `[R|t]` を推定（§4）。真の姿勢と比較。

In [ ]:
i=0
ok,rvec_est,tvec_est=cv2.solvePnP(objpoints[i],imgpoints[i],K_est,dist_est)
R_est,_=cv2.Rodrigues(rvec_est)
R_gt,_=cv2.Rodrigues(poses[i][0])
ang=np.rad2deg(np.arccos((np.trace(R_gt.T@R_est)-1)/2))
print('回転誤差:',round(float(ang),4),'deg')
print('t 真値:',poses[i][1].ravel(),' / 推定:',tvec_est.ravel())
print('カメラ中心 C=-R^T t :', (-R_est.T@tvec_est).ravel())

# 再投影誤差
rep,_=cv2.projectPoints(objpoints[i],rvec_est,tvec_est,K_est,dist_est)
err=np.linalg.norm(rep.reshape(-1,2)-imgpoints[i].reshape(-1,2),axis=1)
print('再投影誤差 平均:',round(err.mean(),4),'px')

---

## 3. ステレオキャリブレーション `stereoCalibrate`

2カメラ間の相対姿勢 `R,T` を推定する（§5.3）。左右で同じボードを同時観測した合成対応点を作る。

In [ ]:
# 真の相対姿勢（cam1 -> cam2）: 主に横方向の基線 + 軽い輻輳
R_rel_gt,_=cv2.Rodrigues(np.array([0.0,np.deg2rad(-3),0.0]))
T_rel_gt=np.array([[0.10],[0.0],[0.0]])     # 基線 10cm
K2_gt=K_gt.copy(); dist2_gt=dist_gt.copy()

objpts=[]; imgL=[]; imgR=[]
for _ in range(N_VIEWS):
    rvec,tvec=random_board_pose()
    Rb,_=cv2.Rodrigues(rvec)
    # cam2でのボード姿勢: X_c2 = R_rel(R_b X + t_b) + T_rel
    Rb2=R_rel_gt@Rb; tb2=R_rel_gt@tvec+T_rel_gt
    rvec2,_=cv2.Rodrigues(Rb2)
    iL,_=cv2.projectPoints(objp,rvec,tvec,K_gt,dist_gt)
    iR,_=cv2.projectPoints(objp,rvec2,tb2,K2_gt,dist2_gt)
    objpts.append(objp.copy())
    imgL.append(iL.reshape(-1,1,2).astype(np.float32))
    imgR.append(iR.reshape(-1,1,2).astype(np.float32))

flags=cv2.CALIB_FIX_INTRINSIC   # 内部は既知として相対姿勢だけ推定
ret,K1,d1,K2,d2,R_rel,T_rel,E,F=cv2.stereoCalibrate(
    objpts,imgL,imgR,K_gt,dist_gt,K2_gt,dist2_gt,IMG_SIZE,flags=flags)
print('ステレオRMS:',round(ret,5),'px')
ang=np.rad2deg(np.arccos((np.trace(R_rel_gt.T@R_rel)-1)/2))
print('相対回転 誤差:',round(float(ang),4),'deg')
print('T 真値:',T_rel_gt.ravel(),' / 推定:',T_rel.ravel())

---

## 4. レクティフィケーション `stereoRectify`

`R1,R2`（各カメラの平行化回転）, `P1,P2`（新射影行列）, `Q`（再投影行列）を得る（§6）。

In [ ]:
R1,R2,P1,P2,Q,roi1,roi2=cv2.stereoRectify(
    K1,d1,K2,d2,IMG_SIZE,R_rel,T_rel,flags=cv2.CALIB_ZERO_DISPARITY,alpha=0)
print('P1=\n',P1)
print('P2=\n',P2,'  ← P2[0,3] = -f*baseline（横ずれ）')
f_rect=P1[0,0]; baseline=-P2[0,3]/P2[0,0]
print(f'レクティファイ後 f={f_rect:.2f}px, baseline={baseline*100:.2f}cm')
print('Q=\n',Q)

In [ ]:
# レクティファイ用 remap マップ（歪み補正＋平行化を統合）
map1x,map1y=cv2.initUndistortRectifyMap(K1,d1,R1,P1,IMG_SIZE,cv2.CV_32FC1)
map2x,map2y=cv2.initUndistortRectifyMap(K2,d2,R2,P2,IMG_SIZE,cv2.CV_32FC1)
print('remap マップ生成完了:',map1x.shape)

# 検証: 対応点をレクティファイ後座標に変換し、左右の v が一致するか
ptsL=imgL[0].reshape(-1,2); ptsR=imgR[0].reshape(-1,2)
rL=cv2.undistortPoints(ptsL.reshape(-1,1,2),K1,d1,R=R1,P=P1).reshape(-1,2)
rR=cv2.undistortPoints(ptsR.reshape(-1,1,2),K2,d2,R=R2,P=P2).reshape(-1,2)
dv=np.abs(rL[:,1]-rR[:,1])
print('レクティファイ後の縦ズレ |v_L - v_R| 平均:',round(dv.mean(),4),'px（≈0で水平整列）')

In [ ]:
plt.figure(figsize=(7,4))
plt.scatter(rL[:,0],rL[:,1],c='b',s=10,label='左')
plt.scatter(rR[:,0],rR[:,1],c='r',s=10,label='右')
for i in range(0,len(rL),5): plt.axhline(rL[i,1],color='gray',lw=0.3)
plt.gca().invert_yaxis(); plt.legend()
plt.title('レクティファイ後: 対応点が同じ行に整列'); plt.xlabel('u'); plt.ylabel('v'); plt.show()

---

## 5. 視差 → 深度（Q による再投影）

レクティファイ後の視差 `d` と再投影行列 `Q` から3D（深度）を復元する（§6.3）。

In [ ]:
# 対応点の視差 d = u_L - u_R から Q で3D化
disp=rL[:,0]-rR[:,0]
# Q: [X,Y,Z,W]^T = Q [u, v, d, 1]^T,  3D = (X,Y,Z)/W
homog=np.c_[rL[:,0],rL[:,1],disp,np.ones(len(rL))]
XYZW=(Q@homog.T).T
XYZ=XYZW[:,:3]/XYZW[:,3:4]
print('視差→深度 Z の範囲:',round(XYZ[:,2].min(),3),'-',round(XYZ[:,2].max(),3),'m')

# 公式 Z=f*B/d とも一致するか
Z_formula=f_rect*baseline/disp
print('Q による Z と f*B/d の最大差:',round(np.abs(XYZ[:,2]-Z_formula).max(),5),'m')

---

## 6. 手を動かす課題

1. **歪みを強める**: `dist_gt` の k1,k2 を大きくし、`calibrateCamera` の推定精度を見る
2. **姿勢数を減らす**: `N_VIEWS=4` にして推定が不安定になる様子を観察（§8の縮退）
3. **基線を変える**: `T_rel_gt` を 0.05/0.30m にし、`Z=fB/d` の深度分解能の違いを計算
4. **ノイズを加える**: imagePointsにガウスノイズを足し、RMS誤差・外部パラメータ誤差の変化を見る
5. **実画像で**: 自分のチェッカーボード画像 → `cv2.findChessboardCorners` → 本ノートと同じ流れ
6. **nuScenes**: `calibrated_sensor` の rotation/translation を読み、camera-LiDAR投影を再現

> 観察を #daily-action-log に記録すると、デイリーレビューの「学習・成長」に反映される。